## Why you almost never write a ReplicaSet directly

ReplicaSets are great at *keeping N pods alive* and terrible at *changing what they run*. Try to update the image on a ReplicaSet and you hit walls:

- Editing `template...image` on an existing ReplicaSet **does not restart any pods.** The new image only applies to pods created *from now on*; existing pods keep the old image.
- To actually roll it out you'd delete pods one by one and let the controller recreate them — annoying with three, error-prone with thirty.
- **No history.** Change the template and the old one is gone — no "go back to last week."

What you want is a controller that owns *multiple* ReplicaSets — one per version — and choreographs the transition, with history and one-command rollback. That's the **Deployment**.

```
   Deployment (v2)          ← you write this
        │ owns
   ┌────┴─────┐
 ReplicaSet v2   ReplicaSet v1
 replicas: 3     replicas: 0   ← kept for rollback
        │ owns
      pod pod pod
```

You write the Deployment. The **Deployment controller** creates a ReplicaSet for the current version; the **ReplicaSet controller** creates the pods. Change the template → the Deployment controller creates a *new* ReplicaSet, scales it up while scaling the old down, and leaves the old at `replicas: 0` so you can roll back.

This three-layer ownership — **Deployment owns ReplicaSets owns Pods** — is one of the most-tested CKA topics. On our map, the **Deployment** chip drives the **ReplicaSet** chip beside it; keep that arrow in mind for the rest of the module.